# Agent Implementation with Langchain


## Implementing an AI Agent with Langgraph & Langchain

In [12]:
!python -m pip install pyowm wikipedia langchain-google-genai --quiet


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [3]:
# initializing the llm
model_name="gemini-2.5-flash"
from langchain.chat_models import init_chat_model
model = init_chat_model(model_name, model_provider="google_genai")


# load a tool
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import OpenWeatherMapQueryRun,WikipediaQueryRun

wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
owm = OpenWeatherMapQueryRun() # make sure to have env variable for OPENWEATHERMAP_API_KEY

tools = [wiki]

# initialize agent
from langchain.agents import create_agent
agent = create_agent(model,tools)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [4]:
from langchain_core.messages import HumanMessage
response = agent.invoke({"messages":HumanMessage(content="hi")})
response['messages']

[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='c1a1df3b-707d-4499-908b-72215b221e43'),
 AIMessage(content="Hello! I'm a large language model, able to answer questions and communicate in response to a wide range of prompts and questions. How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019df69b-802e-76e0-a8fc-22bb2f016e69-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 34, 'total_tokens': 108, 'input_token_details': {'cache_read': 0}})]

In [5]:
response = agent.invoke({"messages":HumanMessage(content="Where is the city Kualalumpur located?")})
response['messages']

[HumanMessage(content='Where is the city Kualalumpur located?', additional_kwargs={}, response_metadata={}, id='d01b6c89-735d-4aed-a209-6c8e2c3d3218'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'wikipedia', 'arguments': '{"query": "Kualalumpur"}'}, '__gemini_function_call_thought_signatures__': {'79ac22a0-bc17-4ee1-b062-752791a85df3': 'CrIDAQw51seE/jHmYmsZGTke+SDvmgGcDVfWadIyZzTls2a7XyedCqk0bQySUmt0lMswOp5lKbRzoGNDcH5o75Mbw+JU7jeHv+tbaGRjJwjSTKDlvlvFb2HWSaCzcx8UHBpp9OstgjbEfeXccFKJTXwcwweKGeKKvzoeCdemtgs8arvv1Im2ld8JRD6HWllkFdGrC7lsJBdIobLmK8IwUykJRqX2Th0Vs2wP0PBByo7JhNgx9Z3B2vIctSRaPYEDX8xll/apE40f11MeMNz37FW+FFO/5CXn/mSdEQcxwhzIuaFGksjuJGP4SQZOH21bhAkScB9/HzIhqtMa3DbSI7bLm2dQ/5moLwIALL8nKAtJXiNuic+QPp2E+8wZjAEZqsIszufp2v8UMtLYwJ1+pLD+uXggkUVUkviVxbeiL9toZIwUmoX02D8LsEIKG697AH2Nv6mS7/iS/w4z/KkzL35l0Q+pkiC/CyTtFyLYpdXgrMUm77y4zA960HZO+UQfUPArqkd1DgjTXnAyGPY6RpXWb9VlT0RD1Fb/1CRD2f1tgPKOZ1JZTqb6HMtsNCP7rT1rw4s='}}, response_metadata={'finish_reason': 'STOP', 'mo

In [6]:
for step in agent.stream({"messages":[HumanMessage(content="Where is the city Kualalumpur located?")]},
                         stream_mode='values'):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Where is the city Kualalumpur located?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (7d3c7a8d-665b-4f10-844a-ef6a3f9058d9)
 Call ID: 7d3c7a8d-665b-4f10-844a-ef6a3f9058d9
  Args:
    query: Kuala Lumpur
================================= Tool Message =================================
Name: wikipedia

Page: Kuala Lumpur
Summary: Kuala Lumpur (KL), officially the Federal Territory of Kuala Lumpur, is the capital city and a federal territory of Malaysia. It is the most populous city in the country, covering an area of 243 km2 (94 sq mi) with a population of 2,075,600 as of 2024. Greater Kuala Lumpur, which itself includes the Klang Valley, is an urban agglomeration of 8.81 million people as of 2024. It is among the fastest growing metropolitan regions in Southeast Asia, in terms of both population and economic development.
The city serves as the cultu

## Enhancing Agent Capabilities
- Adding memory checkpointer
- Adding a middleware for Human in the loop
- Adding a response format for output

In [7]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()



wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
owm = OpenWeatherMapQueryRun() # make sure to have env variable for OPENWEATHERMAP_API_KEY

tools = [wiki,owm]


agent = create_agent(model,tools,checkpointer=memory)

config = {"configurable":{"thread_id":"abc112"}}

for step in agent.stream({"messages":[HumanMessage(content="Where is the city Kualalumpur located?")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Where is the city Kualalumpur located?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (5d380c45-a016-408a-a5ed-20cd20c4b3c4)
 Call ID: 5d380c45-a016-408a-a5ed-20cd20c4b3c4
  Args:
    query: Kualalumpur
================================= Tool Message =================================
Name: wikipedia

Page: Kuala Lumpur
Summary: Kuala Lumpur (KL), officially the Federal Territory of Kuala Lumpur, is the capital city and a federal territory of Malaysia. It is the most populous city in the country, covering an area of 243 km2 (94 sq mi) with a population of 2,075,600 as of 2024. Greater Kuala Lumpur, which itself includes the Klang Valley, is an urban agglomeration of 8.81 million people as of 2024. It is among the fastest growing metropolitan regions in Southeast Asia, in terms of both population and economic development.
The city serves as the cultur

In [8]:
for step in agent.stream({"messages":[HumanMessage(content="I live there.")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

I live there.
================================== Ai Message ==================================

That's interesting! Kuala Lumpur is a vibrant city.


In [9]:
for step in agent.stream({"messages":[HumanMessage(content="Whats the weather there?.")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Whats the weather there?.
================================== Ai Message ==================================
Tool Calls:
  open_weather_map (0ad0dbb9-b990-4e01-a6c8-ac9626e33bef)
 Call ID: 0ad0dbb9-b990-4e01-a6c8-ac9626e33bef
  Args:
    location: Kuala Lumpur,MY
================================= Tool Message =================================
Name: open_weather_map

In Kuala Lumpur,MY, the current weather is as follows:
Detailed status: few clouds
Wind speed: 2.57 m/s, direction: 210°
Humidity: 78%
Temperature: 
  - Current: 32.36°C
  - High: 32.75°C
  - Low: 32.36°C
  - Feels like: 39.36°C
Rain: {}
Heat index: None
Cloud cover: 20%
================================== Ai Message ==================================

The current weather in Kuala Lumpur is few clouds, with a temperature of 32.36°C, feeling like 39.36°C. The wind speed is 2.57 m/s from 210°, and the humidity is 78%.


In [10]:
owm.name

'open_weather_map'

In [11]:
wiki.name

'wikipedia'

### Middleware for HITL

In [12]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

mw1 = HumanInTheLoopMiddleware(interrupt_on={'wikipedia':False,'open_weather_map':True})

agent = create_agent(model=model,
                     tools=[owm,wiki],
                     checkpointer=memory,
                     middleware=[mw1],
                     system_prompt= "you are a helpful assistant")

In [13]:
config = {'configurable':{"thread_id":"1223"}}
for step in agent.stream({"messages":[HumanMessage(content="Tell me about the world war II.")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about the world war II.
================================== Ai Message ==================================
Tool Calls:
  wikipedia (9ee39743-c99d-4a2d-af4e-e620e722faea)
 Call ID: 9ee39743-c99d-4a2d-af4e-e620e722faea
  Args:
    query: World War II
================================= Tool Message =================================
Name: wikipedia

Page: World War II
Summary: World War II, or the Second World War (1 September 1939 – 2 September 1945), was a global conflict between two coalitions: the Allies and the Axis powers. Nearly all of the world's countries participated. Tanks and aircraft played major roles, the latter enabling the strategic bombing of cities and delivery of the only nuclear weapons used in war. World War II was the deadliest conflict in history, causing the death of 60 to 75 million people. Millions died as a result of massacres, starvation, disease, and genocides, including the

In [14]:
for step in agent.stream({"messages":[HumanMessage(content="Whats the weather in delhi today?.")]},
                         stream_mode='values',config=config):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Whats the weather in delhi today?.
================================== Ai Message ==================================
Tool Calls:
  open_weather_map (5d6c8e52-a13d-4bbf-8f82-66c60913f658)
 Call ID: 5d6c8e52-a13d-4bbf-8f82-66c60913f658
  Args:
    location: Delhi,IN
================================== Ai Message ==================================
Tool Calls:
  open_weather_map (5d6c8e52-a13d-4bbf-8f82-66c60913f658)
 Call ID: 5d6c8e52-a13d-4bbf-8f82-66c60913f658
  Args:
    location: Delhi,IN


In [15]:
from langgraph.types import Command
agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)

{'messages': [HumanMessage(content='Tell me about the world war II.', additional_kwargs={}, response_metadata={}, id='4340e3b5-4541-4332-82b4-9c3934f55bc0'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'wikipedia', 'arguments': '{"query": "World War II"}'}, '__gemini_function_call_thought_signatures__': {'9ee39743-c99d-4a2d-af4e-e620e722faea': 'Cv4BAQw51sdAieTgCOttnLjzcLHCboEVn9JEiud/Uri8XpBP3gMIFFzbDzLaE314a3pMogDxW11v9QMBjHBhqAhFE/WlynDYkbMeIYk3bjAiUV6pbQZEz49Yt6TPBIT6oy1xlPohVIKK0Wbs4aV5/KPxY91q/YR0v/ZO+TUXgyP2pkkqKLcOYh7ofyxmZrGuQWO0rtqrhm6pDGivRhbGEOUczejp4iG9rSo0ogtlqznTwtGncp0MLDfKnqoA82PntUpjnF6hn6l924tgm/LBCi4JjGifMby4K3w1ydDcX5/gLQdeg75Hu7b8q5UuQ8RNixFszIz2aPoHK1OrSTFr3U4='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019df6a2-7dca-72d2-a0f6-bed0b9c9e1e9-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'World War II'}, 'id': '9ee39743-c99d-4